> **독립 감사 후 보존용 legacy 노트북입니다.** I3와 Jaccard의 평가 공간이 달랐으며, 아래 필연/확정 표기는 유한 표본 내 공통 배정입니다. 최신 해석과 수정 결과는 `09_independent_audit.ipynb` 및 `outputs/independent_audit/REPORT.md`를 사용하세요.

# 창원국가산단 구간 배정 (Phase 5)

이 노트북은 계산을 새로 하지 않는다. `src/model/revalidation_phase5.py`가 만든
`outputs/tables/*.csv`와 `outputs/report/*.md`를 읽어 표시만 한다. 판정 로직은
이 노트북에 없다. 공식 공간은 `reference_plus_virtual_subspace`(참조사례 3건 +
가상 프로파일 5건 결합 양립공간)다.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from model import config  # noqa: E402

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)
TABLES = ROOT / 'outputs/tables'
FIGS = ROOT / 'outputs/figures'
REPORT = ROOT / 'outputs/report'

## (1) 선호정보 확충 결과

In [ ]:
expansion = pd.read_csv(TABLES / 'electre_preference_expansion.csv')
expansion[expansion.block_type == 'case'][
    ['case_id', 'case_type', 'relation', 'target_stage', 'individual_pass_rate', 'marginal_narrowing']]

**해석**: 8건(실제 3 + 가상 5) 중 어느 사례가 결합 양립공간을 실제로 좁혔는지는
`marginal_narrowing`(이 사례를 뺀 나머지 7건의 교집합 대비, 이 사례를 넣었을 때
줄어드는 비율)으로 판단한다 — 개별 통과율이 낮다고 해서 반드시 공간을 좁히는 것은
아니다(다른 제약과 겹칠 수 있다).

## (2) 공간 3종 비교

In [ ]:
space = expansion[expansion.block_type == 'space']
display(space[['space_id', 'n_samples', 'share_of_full', 'necessary_share_discriminating',
               'mean_n_possible_stages', 'share_n_possible_le_2']])
display(Image(filename=str(FIGS / 'interval_space_narrowing.png')))

**해석**: 참조사례만으로도(1,745표본) 필연배정 비율이 오르지만, 가상 프로파일 5건을
더해 447표본으로 좁히면 판별표본 필연배정 비율이 더 오르고 평균 가능 단계 수가
줄어든다 — 선호정보가 늘수록 배정이 더 많이 결정된다는 것을 보여준다.

## (3) 구간 배정 표

In [ ]:
interval = pd.read_csv(TABLES / 'electre_interval_assignment.csv')
interval[['industry', 'quarter', 'necessary_stage', 'possible_stages', 'n_possible',
         'display_label', 'is_discriminating']].head(15)

**해석**: `display_label`은 `src/model/revalidation_phase5.py`의 `display_label` 함수가
고정한 표기이며, 단일 단계만 적힌 경우는 필연배정(`n_possible==1`)일 때뿐이다.

## (4) 2026Q2 진단카드

In [ ]:
cards_text = (REPORT / 'interval_diagnostic_cards_2026Q2.md').read_text(encoding='utf-8')
first_card_end = cards_text.find('\n## ', cards_text.find('\n## ') + 1)
display(Markdown(cards_text[:first_card_end]))

**해석**: 전체 10개 업종 카드는 `outputs/report/interval_diagnostic_cards_2026Q2.md`에 있다(위는 첫 카드만 예시로 표시).

## (5) 교란 강건성 3지표

In [ ]:
robustness = pd.read_csv(TABLES / 'electre_interval_robustness.csv')
robustness[['model_id', 'scenario_id', 'metric_id', 'value', 'threshold', 'pass',
           'n_replicates', 'mean_n_possible_stages']]

**해석**: C3(점 배정 유지율)는 이미 합격선에 못 미쳤다. C3b(교란 후 점 단계가 원 가능
단계 범위 안에 머무는 비율)는 개선됐지만 여전히 합격선 미달이며, `mean_n_possible_stages`를
함께 봐야 한다 — 가능 단계가 3개인 행은 어떤 교란이 와도 자동으로 포함되기 때문이다.
C3c(가능 단계 집합 자체의 Jaccard 안정성)는 합격선을 통과한다.

## (6) 최종 판정

In [ ]:
decision = pd.read_csv(TABLES / 'electre_final_decision.csv')
display(decision[decision.block_type == 'point_assignment'][
    ['model_id', 'scenario_id', 'c1_pass', 'c2_pass', 'c3_pass', 'c4_pass', 's1_pass', 'overall_status']])
display(decision[decision.block_type == 'interval_assignment'][
    ['i1_value', 'i1_pass', 'i2_value', 'i2_pass', 'i3_value', 'i3_pass', 'i4_value', 'i4_pass',
     'overall_status', 'rationale']])

**해석**: 점 배정 6행은 참고용이며 공식이 아니다(`retained_as_official=False`). 공식
판정은 구간 배정 1행이며, I1·I2·I4는 통과하지만 I3(C3b 평균)가 합격선에 못 미쳐
전체 상태는 미달로 기록된다.

## (7) 한계 기록

In [ ]:
limitation = pd.read_csv(TABLES / 'electre_limitation_record.csv')
print(limitation['statement'].iloc[0])
display(Image(filename=str(FIGS / 'c3_boundary_vs_revision.png')))

**해석**: 이 한계는 파라미터를 더 고른다고 해결되지 않는다 — 경계값과 자료 개정폭의
크기 관계에서 오는 구조적 제약이며, `outputs/tables/electre_limitation_record.csv`에
실측값과 함께 고정 기록됐다.